### setup

In [1]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/workspace/competitions/Sly/Chatbot_HyperTension/data/1.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

105


In [3]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Circulation
September 16, 2025 Circulation. 2025;152:e114–e218. DOI: 10.1161/CIR.0000000000001356
Circulation is available at www.ahajournals.org/journal/circe114
 
*Writing committee members are requ

{'producer': 'Adobe PDF Library 15.0; modified using iText 4.2.0 by 1T3XT', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2025-09-06T13:43:49+05:30', 'moddate': '2025-10-28T22:33:40-07:00', 'trapped': '/False', 'subject': 'Circulation 2025.152:e114-e218', 'author': 'Daniel W. Jones', 'title': '2025 AHA/ACC/AANP/AAPA/ABC/ACCP/ACPM/AGS/AMA/ASPC/NMA/PCNA/SGIM Guideline for the Prevention, Detection, Evaluation and Management of High Blood Pressure in Adults: A Report of the American College of Cardiology/American Heart Association Joint Committee on Clinical Practice Guidelines', 'source': '/workspace/competitions/Sly/Chatbot_HyperTension/data/1.pdf', 'total_pages': 105, 'page': 0, 'page_label': 'e114'}


### 2. Splitter


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

767


### Embedding

In [6]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [7]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 3072

[0.010719627141952515, 0.02934115193784237, -0.010306156240403652, 0.008223485201597214, 0.03399653360247612, -0.007633905857801437, -0.011699707247316837, 0.04094897583127022, -0.005459352862089872, 0.004406532738357782]


### Vector Store:

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
ids = vector_store.add_documents(documents=all_splits)

### Test vector store

In [10]:
results = vector_store.similarity_search(
    "What is blood pressure?"
)

print(results[0])

page_content='Synopsis
Historically, the measurement of BP in the office setting 
was performed by using auscultatory BP measurements 
using a calibrated mercury column, which was later re-
placed by auscultatory measurements using a nonmercury 
Figure 2. Estimated 10-Year Risks for 
Total Cardiovascular Disease Using 
the PREVENTTM CVD Risk Equations, 
Stratified by Blood Pressure Levels 
With Selected Combinations of Risk 
Factors.
CVD indicates cardiovascular disease; 
eGFR, estimated glomerular filtration rate; 
and HDL, high-density lipoprotein. Derived 
from Khan et al19,20 via PREVENT.
Downloaded from http://ahajournals.org by on October 28, 2025' metadata={'producer': 'Adobe PDF Library 15.0; modified using iText 4.2.0 by 1T3XT', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2025-09-06T13:43:49+05:30', 'moddate': '2025-10-28T22:33:40-07:00', 'trapped': '/False', 'subject': 'Circulation 2025.152:e114-e218', 'author': 'Daniel W. Jones', 'title': '2025 AHA/ACC/AANP/

In [11]:
results

[Document(id='abb18f19-5468-4362-805b-2b889e1839b7', metadata={'producer': 'Adobe PDF Library 15.0; modified using iText 4.2.0 by 1T3XT', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2025-09-06T13:43:49+05:30', 'moddate': '2025-10-28T22:33:40-07:00', 'trapped': '/False', 'subject': 'Circulation 2025.152:e114-e218', 'author': 'Daniel W. Jones', 'title': '2025 AHA/ACC/AANP/AAPA/ABC/ACCP/ACPM/AGS/AMA/ASPC/NMA/PCNA/SGIM Guideline for the Prevention, Detection, Evaluation and Management of High Blood Pressure in Adults: A Report of the American College of Cardiology/American Heart Association Joint Committee on Clinical Practice Guidelines', 'source': '/workspace/competitions/Sly/Chatbot_HyperTension/data/1.pdf', 'total_pages': 105, 'page': 12, 'page_label': 'e126', 'start_index': 1604}, page_content='Synopsis\nHistorically, the measurement of BP in the office setting \nwas performed by using auscultatory BP measurements \nusing a calibrated mercury column, which was later r

### 4. Retriever:

In [12]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[[Document(id='99d80333-79d3-4257-bb2c-bcf1b77d5d40', metadata={'producer': 'Adobe PDF Library 15.0; modified using iText 4.2.0 by 1T3XT', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2025-09-06T13:43:49+05:30', 'moddate': '2025-10-28T22:33:40-07:00', 'trapped': '/False', 'subject': 'Circulation 2025.152:e114-e218', 'author': 'Daniel W. Jones', 'title': '2025 AHA/ACC/AANP/AAPA/ABC/ACCP/ACPM/AGS/AMA/ASPC/NMA/PCNA/SGIM Guideline for the Prevention, Detection, Evaluation and Management of High Blood Pressure in Adults: A Report of the American College of Cardiology/American Heart Association Joint Committee on Clinical Practice Guidelines', 'source': '/workspace/competitions/Sly/Chatbot_HyperTension/data/1.pdf', 'total_pages': 105, 'page': 99, 'page_label': 'e213', 'start_index': 796}, page_content='• CANP*\n • Case Western Reserve School of \nNursing\n • College of Nursing Pennsylvania \nState University*\n • Delaware Health Force*\n • Eck Institute for Global Health, \nU

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[[Document(id='99d80333-79d3-4257-bb2c-bcf1b77d5d40', metadata={'producer': 'Adobe PDF Library 15.0; modified using iText 4.2.0 by 1T3XT', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2025-09-06T13:43:49+05:30', 'moddate': '2025-10-28T22:33:40-07:00', 'trapped': '/False', 'subject': 'Circulation 2025.152:e114-e218', 'author': 'Daniel W. Jones', 'title': '2025 AHA/ACC/AANP/AAPA/ABC/ACCP/ACPM/AGS/AMA/ASPC/NMA/PCNA/SGIM Guideline for the Prevention, Detection, Evaluation and Management of High Blood Pressure in Adults: A Report of the American College of Cardiology/American Heart Association Joint Committee on Clinical Practice Guidelines', 'source': '/workspace/competitions/Sly/Chatbot_HyperTension/data/1.pdf', 'total_pages': 105, 'page': 99, 'page_label': 'e213', 'start_index': 796}, page_content='• CANP*\n • Case Western Reserve School of \nNursing\n • College of Nursing Pennsylvania \nState University*\n • Delaware Health Force*\n • Eck Institute for Global Health, \nU

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
